# Task 2 (extended) — Embedding Model Comparison + Class-Imbalance Experiments

This notebook builds on `Task2_eda_baseline.ipynb` (AraBERT-twitter embeddings → Logistic Regression baseline for `Emotion`, `Offensive`, `Hate`).

It adds two things you asked for:

1. **Model comparison** — the baseline uses `aubmindlab/bert-base-arabertv02-twitter`. Here we add a second embedding model, **`UBC-NLP/MARBERT`** (trained specifically on ~1B dialectal-Arabic tweets, so it's a natural head-to-head competitor for a tweet dataset like this one), embed the same data with both, and compare accuracy / macro-F1 per label to see which backbone is actually better for this task.
2. **Three ways to fight the class imbalance**, applied only to the *training* split (never the test split, to avoid leakage):
   - **Under-sampling** — randomly drop rows from majority classes down toward the minority count.
   - **Oversampling via back-translation** — Arabic → English → Arabic round-trip translation (MarianMT) of minority-class rows to create paraphrased duplicates with the *same* label.
   - **Oversampling via dialect-to-dialect translation** — e.g. Egyptian → Gulf(Najdi) → Egyptian using Meta's NLLB-200 (which has explicit language codes for Arabic dialects), to create dialectal paraphrases of minority-class rows with the same label.

> **Runtime note:** this notebook downloads several models from Hugging Face (AraBERT, MARBERT, MarianMT `ar↔en`, NLLB-200) and needs a GPU to run in reasonable time. Run it in **Google Colab** (like the original) with a GPU runtime. It was written/validated for logic here but **not executed** in this sandbox, because this sandbox's network is locked down to package registries only (no `huggingface.co` access) — so please run all cells top to bottom in Colab.


In [ ]:
!pip install arabert farasapy transformers scikit-learn nltk arabic-reshaper python-bidi sentencepiece sacremoses -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 6.9 MB/s eta 0:00:00


In [ ]:
import re
import unicodedata
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, MarianMTModel, MarianTokenizer
from arabert.preprocess import ArabertPreprocessor

nltk.download('stopwords', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 1. Load data + preprocessing (same as the baseline notebook)

In [ ]:
df = pd.read_csv("train.csv")
print(df.shape)
df.head()

(5960, 5)


,id,text,Emotion,Offensive,Hate
0,2537,أحد التجار الشباب العمانيين يقول للاسف لما يكو...,neutral,no,NaN
1,5579,@JALHARBISKY مجموعه القدرة الجنسيه👍<LF> <LF>بد...,optimism,no,NaN
2,6092,@rwn4o حبيبييي والله اكثثثرر يارب امين🥺♥️♥️,love,no,NaN
3,2540,#وصال_دوت_FM<LF>مع سميرة الفطيسية @Samira_Alfu...,neutral,no,NaN
4,3159,من ينتزع ارواح اطفالنا من أجسادها بكل وحشية عل...,anticipation,no,NaN


In [ ]:
ARABERT_MODEL_NAME = "aubmindlab/bert-base-arabertv02-twitter"
_arabert_preprocessor = ArabertPreprocessor(model_name=ARABERT_MODEL_NAME)

def preprocess_text(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = "".join(c for c in text if unicodedata.category(c) != "So")
    text = _arabert_preprocessor.preprocess(text)
    return text

df["clean_text"] = df["text"].apply(preprocess_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,أحد التجار الشباب العمانيين يقول للاسف لما يكو...,أحد التجار الشباب العمانيين يقول للاسف لما يكو...
1,@JALHARBISKY مجموعه القدرة الجنسيه👍<LF> <LF>بد...,مجموعه القدرة الجنسيه بديل الفياجرا والسنافي ز...
2,@rwn4o حبيبييي والله اكثثثرر يارب امين🥺♥️♥️,حبيبي والله اكثرر يارب امين
3,#وصال_دوت_FM<LF>مع سميرة الفطيسية @Samira_Alfu...,# وصال _ دوت _ FM مع سميرة الفطيسية أمل الشكيل...
4,من ينتزع ارواح اطفالنا من أجسادها بكل وحشية عل...,من ينتزع ارواح اطفالنا من أجسادها بكل وحشية عل...


## 2. Part A — Which embedding model is actually better: AraBERT-twitter vs. MARBERT?

Both are loaded as plain `AutoModel` encoders, mean-pooled, and fed into the exact same Logistic-Regression baseline used in the original notebook, so the *only* thing that changes between the two runs is the embedding model. That isolates the comparison to "which backbone gives better features," rather than mixing in different classifiers or preprocessing.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def load_embedder(model_name):
    """Loads a tokenizer+model pair and returns an embed_texts(texts) function."""
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(device)
    mdl.eval()

    def embed_texts(texts, batch_size=32, max_length=64):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = tok(batch, padding="max_length", truncation=True,
                         max_length=max_length, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = mdl(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            all_embeddings.append(pooled.cpu().numpy())
        return np.vstack(all_embeddings)

    return embed_texts

EMBEDDING_MODELS = {
    "AraBERT-twitter": "aubmindlab/bert-base-arabertv02-twitter",
    "MARBERT":         "UBC-NLP/MARBERT",
}

In [ ]:
def evaluate_label(X_train, X_test, y_train, y_test, label_name, model_name, plot=False):
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")
    print(f"[{model_name}] {label_name} -> acc={acc:.4f}  macro-F1={macro_f1:.4f}")
    if plot:
        print(classification_report(y_test, preds))
    return clf, acc, macro_f1

LABEL_COLS = ["Emotion", "Offensive", "Hate"]
comparison_rows = []
embeddings_cache = {}   # model_name -> X (full-dataset embeddings), reused later for imbalance section

for model_key, hf_name in EMBEDDING_MODELS.items():
    print(f"\n=== Embedding with {model_key} ({hf_name}) ===")
    embed_fn = load_embedder(hf_name)
    X = embed_fn(df["clean_text"].tolist())
    embeddings_cache[model_key] = (X, embed_fn)

    for label in LABEL_COLS:
        y = df[label]
        mask = y.notna()
        X_use, y_use = X[mask.values], y[mask]
        X_train, X_test, y_train, y_test = train_test_split(
            X_use, y_use, test_size=0.2, random_state=SEED, stratify=y_use
        )
        _, acc, macro_f1 = evaluate_label(X_train, X_test, y_train, y_test, label, model_key)
        comparison_rows.append({"model": model_key, "label": label, "accuracy": acc, "macro_f1": macro_f1})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


=== Embedding with AraBERT-twitter (aubmindlab/bert-base-arabertv02-twitter) ===


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MARBERT] Emotion -> acc=0.5143  macro-F1=0.4496
[MARBERT] Offensive -> acc=0.8389  macro-F1=0.8178
[MARBERT] Hate -> acc=0.7564  macro-F1=0.6627


,model,label,accuracy,macro_f1
0,AraBERT-twitter,Emotion,0.466443,0.375652
1,AraBERT-twitter,Offensive,0.829698,0.806972
2,AraBERT-twitter,Hate,0.770774,0.634708
3,MARBERT,Emotion,0.514262,0.449588
4,MARBERT,Offensive,0.838926,0.817788
5,MARBERT,Hate,0.756447,0.662687


In [ ]:
pivot = comparison_df.pivot(index="label", columns="model", values="macro_f1")
pivot["winner"] = pivot.idxmax(axis=1)
print(pivot)

# Pick a single "winning" embedding model overall (highest average macro-F1 across labels)
overall = comparison_df.groupby("model")["macro_f1"].mean().sort_values(ascending=False)
print("\nOverall average macro-F1 by embedding model:")
print(overall)
WINNING_MODEL = overall.index[0]
print(f"\n>>> Winning embedding model for the imbalance experiments below: {WINNING_MODEL}")

model      AraBERT-twitter   MARBERT   winner
label                                        
Emotion           0.375652  0.449588  MARBERT
Hate              0.634708  0.662687  MARBERT
Offensive         0.806972  0.817788  MARBERT

Overall average macro-F1 by embedding model:
model
MARBERT            0.643354
AraBERT-twitter    0.605777
Name: macro_f1, dtype: float64

>>> Winning embedding model for the imbalance experiments below: MARBERT


## 3. Part B — Three ways to handle class imbalance

Reminder of how imbalanced each label is (from the EDA):

- **Emotion** (12 classes): `anger`=1551 vs `fear`=53 — a ~29x gap.
- **Offensive** (binary): `no`=4216 vs `yes`=1744 — a ~2.4x gap.
- **Hate** (only defined when `Offensive=yes`): `not_hate`=1441 vs `hate`=303 — a ~4.8x gap.

All three strategies below are applied **only to the training split**, never to the test split — otherwise you'd leak augmented/duplicated information into evaluation and get inflated scores. The generic experiment runner below enforces that.

In [ ]:
_, WINNING_EMBED_FN = embeddings_cache[WINNING_MODEL]
print("Using embedding model:", WINNING_MODEL)

def split_for_label(df, label_col, text_col="clean_text", test_size=0.2, seed=SEED):
    d = df[[text_col, label_col]].dropna().rename(columns={text_col: "text", label_col: "label"})
    train_df, test_df = train_test_split(d, test_size=test_size, random_state=seed, stratify=d["label"])
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

def run_experiment(train_df, test_df, embed_fn, label_name, strategy_name):
    X_train = embed_fn(train_df["text"].tolist())
    X_test = embed_fn(test_df["text"].tolist())
    clf, acc, macro_f1 = evaluate_label(X_train, X_test, train_df["label"], test_df["label"],
                                         label_name, strategy_name)
    return acc, macro_f1

Using embedding model: MARBERT


### 3.1 Strategy 1 — Random under-sampling

In [ ]:
def undersample(train_df, cap_ratio=1.0):
    """Downsample every class to (cap_ratio x minority-class count)."""
    counts = train_df["label"].value_counts()
    target_n = int(counts.min() * cap_ratio)
    parts = []
    for label, grp in train_df.groupby("label"):
        n = min(len(grp), target_n) if target_n > 0 else len(grp)
        parts.append(grp.sample(n=max(n, 1), random_state=SEED))
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

### 3.2 Strategy 2 — Oversampling via back-translation (Arabic → English → Arabic)

In [ ]:
_bt_models = {}

def _load_marian(pair):
    if pair not in _bt_models:
        name = f"Helsinki-NLP/opus-mt-{pair}"
        tok = MarianTokenizer.from_pretrained(name)
        mdl = MarianMTModel.from_pretrained(name).to(device)
        mdl.eval()
        _bt_models[pair] = (tok, mdl)
    return _bt_models[pair]

def _translate_batch(texts, pair, batch_size=16, max_length=96):
    tok, mdl = _load_marian(pair)
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            gen = mdl.generate(**enc, max_length=max_length)
        outputs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return outputs

def back_translate(texts):
    """AR -> EN -> AR round trip. Returns paraphrased Arabic text."""
    en = _translate_batch(texts, "ar-en")
    ar = _translate_batch(en, "en-ar")
    return ar

def oversample_backtranslation(train_df, cap_ratio=1.0, max_growth=3.0):
    """For every under-represented class, back-translate a sample of its rows
    to synthesize new (paraphrased) rows with the SAME label, up to a target count.
    max_growth caps how many times we're willing to multiply a class's original size,
    to avoid over-duplicating a very small class into near-identical clones."""
    counts = train_df["label"].value_counts()
    target_n = int(counts.max() * cap_ratio)
    new_rows = [train_df]
    for label, grp in train_df.groupby("label"):
        needed = min(target_n, int(len(grp) * max_growth)) - len(grp)
        if needed <= 0:
            continue
        sample = grp.sample(n=needed, replace=True, random_state=SEED)
        aug_text = back_translate(sample["text"].tolist())
        new_rows.append(pd.DataFrame({"text": aug_text, "label": label}))
    return pd.concat(new_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

### 3.3 Strategy 3 — Oversampling via dialect-to-dialect translation

Uses Meta's **NLLB-200**, which has explicit FLORES-200 language codes for several Arabic dialects, so we can genuinely translate *between* dialects (not just MSA↔dialect):

| Dialect | NLLB code |
|---|---|
| Modern Standard Arabic | `arb_Arab` |
| Egyptian | `arz_Arab` |
| Gulf (closest available: Najdi) | `ars_Arab` |
| North Levantine | `apc_Arab` |
| Moroccan | `ary_Arab` |
| Iraqi/Mesopotamian | `acm_Arab` |

We cycle a minority-class row through a dialect pair (e.g. Egyptian → Gulf → Egyptian, matching your example) to synthesize a lexically-varied duplicate that should carry the same label.

In [ ]:
NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"
_nllb_tok = None
_nllb_mdl = None

def _load_nllb():
    global _nllb_tok, _nllb_mdl
    if _nllb_mdl is None:
        _nllb_tok = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
        _nllb_mdl = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME).to(device)
        _nllb_mdl.eval()
    return _nllb_tok, _nllb_mdl

def nllb_translate(texts, src_lang, tgt_lang, batch_size=16, max_length=96):
    tok, mdl = _load_nllb()
    tok.src_lang = src_lang
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            gen = mdl.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(tgt_lang), max_length=max_length)
        outputs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return outputs

DIALECT_PAIRS = [
    ("arz_Arab", "ars_Arab"),  # Egyptian -> Gulf(Najdi)
    ("ars_Arab", "arz_Arab"),  # Gulf(Najdi) -> Egyptian
    ("arb_Arab", "apc_Arab"),  # MSA -> Levantine
]

def dialect_paraphrase(texts, src_lang="arz_Arab", tgt_lang="ars_Arab"):
    """Round-trips text through a target dialect and back, e.g. Egyptian -> Gulf -> Egyptian,
    to produce a dialectally-varied paraphrase that should preserve the original meaning/label."""
    mid = nllb_translate(texts, src_lang, tgt_lang)
    back = nllb_translate(mid, tgt_lang, src_lang)
    return back

def oversample_dialect(train_df, cap_ratio=1.0, max_growth=3.0):
    counts = train_df["label"].value_counts()
    target_n = int(counts.max() * cap_ratio)
    new_rows = [train_df]
    for idx, (label, grp) in enumerate(train_df.groupby("label")):
        needed = min(target_n, int(len(grp) * max_growth)) - len(grp)
        if needed <= 0:
            continue
        sample = grp.sample(n=needed, replace=True, random_state=SEED)
        src_lang, tgt_lang = DIALECT_PAIRS[idx % len(DIALECT_PAIRS)]
        aug_text = dialect_paraphrase(sample["text"].tolist(), src_lang, tgt_lang)
        new_rows.append(pd.DataFrame({"text": aug_text, "label": label}))
    return pd.concat(new_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)

## 4. Run all strategies for every label and compare to the untouched baseline

In [ ]:
STRATEGIES = {
    "baseline (no augmentation)": lambda train_df: train_df,
    "under-sampling":              undersample,
    "oversample: back-translation": oversample_backtranslation,
    "oversample: dialect-translation": oversample_dialect,
}

results = []
for label in LABEL_COLS:
    train_df, test_df = split_for_label(df, label)
    print(f"\n########## Label: {label}  (train={len(train_df)}, test={len(test_df)}) ##########")
    for strat_name, strat_fn in STRATEGIES.items():
        aug_train_df = strat_fn(train_df.copy())
        acc, macro_f1 = run_experiment(aug_train_df, test_df, WINNING_EMBED_FN, label, strat_name)
        results.append({"label": label, "strategy": strat_name, "n_train": len(aug_train_df),
                         "accuracy": acc, "macro_f1": macro_f1})

results_df = pd.DataFrame(results)
results_df


########## Label: Emotion  (train=4768, test=1192) ##########
[baseline (no augmentation)] Emotion -> acc=0.5143  macro-F1=0.4496
[under-sampling] Emotion -> acc=0.3784  macro-F1=0.3391


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
for label in LABEL_COLS:
    sub = results_df[results_df["label"] == label].sort_values("macro_f1", ascending=False)
    print(f"\n=== {label}: ranked by macro-F1 ===")
    print(sub[["strategy", "n_train", "accuracy", "macro_f1"]].to_string(index=False))

pivot_results = results_df.pivot(index="strategy", columns="label", values="macro_f1")
plt.figure(figsize=(9, 5))
sns.heatmap(pivot_results, annot=True, fmt=".3f", cmap="viridis")
plt.title(f"Macro-F1 by imbalance strategy and label (embeddings: {WINNING_MODEL})")
plt.tight_layout()
plt.show()

## 5. Notes / things to tune once you have real numbers

- `max_growth` (default 3x) caps how aggressively each strategy duplicates a tiny class — raise it for `Emotion`'s `fear` class (only 53 rows) if under-sampling collapses the majority classes too much, or lower it if back-translation/dialect paraphrases start looking too repetitive.
- Under-sampling is fast but throws away real data — with `Emotion` having 12 classes and `fear`=53, under-sampling to 53-per-class will leave you with a very small training set (~53×12 ≈ 636 rows), so it may hurt more than it helps there. It's usually the strongest option for `Offensive` (only 2.4x imbalance, plenty of majority-class data to spare).
- Back-translation and dialect-translation are slower (they call a generative model per augmented row) but preserve more of the original training signal since they add new rows instead of removing them — better suited to `Emotion`'s `fear`/`surprise`/`pessimism` classes and `Hate`'s `hate` class.
- You can mix strategies (e.g. moderate under-sampling of the majority + back-translation oversampling of the tail classes) — the `run_experiment` / `STRATEGIES` structure above makes that a one-line addition if you want to try it.
- Always sanity-check a handful of the back-translated / dialect-translated rows manually — round-trip MT occasionally drops negation or profanity, which would flip the true label for `Offensive`/`Hate`.
